# StateGen Experiment Results

Compares four code generation strategies on BigCodeBench (hard subset):

| Method | Description |
|---|---|
| `direct_gen` | Single LLM call, no retries |
| `self_planning` | Plan then generate, global retry on failure |
| `self_debugging` | Generate → explain → repair whole program |
| `stategen` | MDP decomposition with local state-level backtracking |

**Primary question:** Can StateGen match or exceed baseline pass@1 while using fewer tokens?

In [1]:
import json
import os
import sys
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── paths ──────────────────────────────────────────────────────────
ROOT = Path(".").resolve().parent  # notebook lives in notebooks/
if not (ROOT / "results").exists():
    ROOT = Path(".").resolve()     # fallback: run from project root
RESULTS = ROOT / "results"

print(f"Results directory: {RESULTS}")
print("Files:", [p.name for p in RESULTS.glob("*") if p.is_file()])

Results directory: /Users/mac/Documents/cmu/spring26/intro-deep-learning/project/stategen/results
Files: ['metrics.json', 'plot_input_output_tokens.png', 'self_planning_solutions.jsonl', 'plot_token_distribution.png', 'plot_retries.png', 'plot_overhead.png', 'plot_pass_vs_tokens.png', 'plot_wall_time.png', 'token_events.jsonl', 'plot_task_heatmap.png', 'plot_memory_reuse.png', 'stategen_solutions.jsonl', 'plot_call_type_tokens.png', 'self_debugging_solutions.jsonl', 'direct_gen_solutions.jsonl']


In [2]:
# ── style ──────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.15)
plt.rcParams.update({"figure.dpi": 130, "figure.facecolor": "white"})

METHOD_ORDER  = ["direct_gen", "self_planning", "self_debugging", "stategen"]
METHOD_LABELS = {
    "direct_gen":    "Direct Gen",
    "self_planning":  "Self-Planning",
    "self_debugging": "Self-Debugging",
    "stategen":       "StateGen (ours)",
}
PALETTE = {
    "direct_gen":    "#7eb0d5",
    "self_planning":  "#fd7f6f",
    "self_debugging": "#b2e061",
    "stategen":       "#bd7ebe",
}

In [3]:
# ── load per-task solution records ─────────────────────────────────
records = []
for path in RESULTS.glob("*_solutions.jsonl"):
    method = path.stem.replace("_solutions", "")
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                rec = json.loads(line)
                rec.setdefault("method", method)
                records.append(rec)

df_all = pd.DataFrame(records)

# ── restrict to the common task set across all methods ──────────────
# (self_debugging timed out on 3 tasks; all 3 fail on every other method)
task_sets = df_all.groupby("method")["task_id"].apply(set)
common_tasks = set.intersection(*task_sets.values)
df = df_all[df_all["task_id"].isin(common_tasks)].copy()

n_all    = df_all["task_id"].nunique()
n_common = len(common_tasks)
dropped  = n_all - n_common
print(f"Total unique tasks: {n_all}  |  Common to all methods: {n_common}  |  Dropped: {dropped}")
if dropped:
    dropped_ids = set(df_all["task_id"].unique()) - common_tasks
    print(f"  Excluded: {sorted(dropped_ids)}")
print(f"Methods: {sorted(df['method'].unique())}")
df.head(3)

Total unique tasks: 148  |  Common to all methods: 145  |  Dropped: 3
  Excluded: ['BigCodeBench/108', 'BigCodeBench/120', 'BigCodeBench/123']
Methods: ['direct_gen', 'self_debugging', 'self_planning', 'stategen']


,task_id,solution,method,total_input_tokens,total_output_tokens,total_tokens,num_attempts,num_retries,wall_time,attempt_details
0,BigCodeBench/13,import subprocess\nimport ftplib\nimport os\n\...,self_planning,1029,475,1504,2,0,36.153325,"[{'attempt': 0, 'passed': True, 'error': '', '..."
1,BigCodeBench/15,import subprocess\nimport csv\nimport os\n\nde...,self_planning,1527,1260,2787,4,2,75.207793,"[{'attempt': 0, 'passed': False, 'error': 'Ass..."
2,BigCodeBench/17,import subprocess\nimport psutil\nimport time\...,self_planning,1595,1089,2684,6,4,61.336942,"[{'attempt': 0, 'passed': False, 'error': '? ..."


In [4]:
# ── load token events (per LLM call) ───────────────────────────────
events = []
tok_path = RESULTS / "token_events.jsonl"
if tok_path.exists():
    with open(tok_path) as f:
        for line in f:
            line = line.strip()
            if line:
                events.append(json.loads(line))
df_events = pd.DataFrame(events) if events else pd.DataFrame()
print(f"Token events: {len(df_events)} rows")
if not df_events.empty:
    print(df_events.groupby(["method", "call_type"]).size().unstack(fill_value=0))

Token events: 1891 rows
call_type       decomposition  explanation  final_repair  generation  \
method                                                                 
direct_gen                  0            0             0         148   
self_debugging              0          219             0         148   
self_planning               0            0             0         373   
stategen                  148            0           254           0   

call_type       planning  repair  
method                            
direct_gen             0       0  
self_debugging         0     219  
self_planning        373       0  
stategen               0       9  


In [5]:
# ── load official eval status if available ──────────────────────────
# Each method can have results from BCB's Docker evaluator:
#   results/{method}_solutions-sanitized-calibrated_eval_results.json
eval_status = {}   # task_id → "pass" | "fail" | ...
for path in RESULTS.glob("*_eval_results.json"):
    with open(path) as f:
        data = json.load(f)
    for task_id, results in data.get("eval", {}).items():
        if results:
            eval_status[task_id] = results[0]["status"]

print(f"Official eval results loaded for {len(eval_status)} tasks" if eval_status
      else "No official eval results yet — using quick_check proxy for pass@1")

No official eval results yet — using quick_check proxy for pass@1


In [6]:
# ── compute per-task pass column ────────────────────────────────────
def task_passed(row):
    if eval_status:
        return eval_status.get(row["task_id"], "fail") == "pass"
    # Fallback: last attempt_details entry
    details = row.get("attempt_details", [])
    if not details:
        return False
    last = details[-1]
    return bool(
        last.get("passed") or
        last.get("passed_quick_check") or
        last.get("passed_bcb")
    )

df["passed"] = df.apply(task_passed, axis=1)
df["total_tokens"] = df.get("total_tokens", df["total_input_tokens"] + df["total_output_tokens"])

# filter to methods we care about and present in data
avail = [m for m in METHOD_ORDER if m in df["method"].values]
df = df[df["method"].isin(avail)].copy()

summary = (
    df.groupby("method")
    .agg(
        n_tasks=("task_id", "count"),
        pass_at_1=("passed", "mean"),
        avg_tokens=("total_tokens", "mean"),
        median_tokens=("total_tokens", "median"),
        avg_retries=("num_retries", "mean"),
        avg_wall_time=("wall_time", "mean"),
    )
    .reindex([m for m in avail])
)
summary["label"] = summary.index.map(METHOD_LABELS)
print(summary[["n_tasks", "pass_at_1", "avg_tokens", "avg_retries", "avg_wall_time"]].to_string())

                n_tasks  pass_at_1   avg_tokens  avg_retries  avg_wall_time
method                                                                     
direct_gen          145   0.234483   603.593103     0.000000      18.980656
self_planning       145   0.289655  3178.068966     3.020690      86.794611
self_debugging      145   0.303448  6560.275862     1.475862     140.014758
stategen            145   0.296552  3093.441379     1.800000      74.370305


## 1. Pass@1 vs Token Cost

The primary trade-off: correctness vs. token cost.

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

methods = [m for m in avail]
labels  = [METHOD_LABELS[m] for m in methods]
colors  = [PALETTE[m] for m in methods]

# ── (a) pass@1 ──────────────────────────────────────────────────────
ax = axes[0]
bars = ax.bar(labels, summary.loc[methods, "pass_at_1"] * 100, color=colors,
              edgecolor="white", linewidth=0.8)
ax.set_title("Pass@1 (%)", fontweight="bold")
ax.set_ylabel("Pass rate (%)")
ax.set_ylim(0, max(summary["pass_at_1"].max() * 120, 15))
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
for bar, val in zip(bars, summary.loc[methods, "pass_at_1"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f"{val:.1%}", ha="center", va="bottom", fontsize=10)
ax.tick_params(axis="x", labelrotation=15)

# ── (b) avg tokens ───────────────────────────────────────────────────
ax = axes[1]
bars = ax.bar(labels, summary.loc[methods, "avg_tokens"], color=colors,
              edgecolor="white", linewidth=0.8)
ax.set_title("Avg Tokens per Task", fontweight="bold")
ax.set_ylabel("Tokens")
for bar, val in zip(bars, summary.loc[methods, "avg_tokens"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 20,
            f"{val:,.0f}", ha="center", va="bottom", fontsize=10)
ax.tick_params(axis="x", labelrotation=15)

plt.suptitle("Pass Rate vs Token Cost", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(RESULTS / "plot_pass_vs_tokens.png", bbox_inches="tight")
plt.show()

/var/folders/c4/98dp1lqj4kq9n2s3k05mvq7m0000gn/T/ipykernel_80898/964148544.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Per-Task Token Distribution

Box plots show how token costs vary across tasks (outliers indicate hard problems or excessive retrying).

In [8]:
fig, ax = plt.subplots(figsize=(9, 4.5))

plot_df = df[df["method"].isin(avail)].copy()
plot_df["Method"] = plot_df["method"].map(METHOD_LABELS)
order = [METHOD_LABELS[m] for m in avail]
pal   = {METHOD_LABELS[m]: PALETTE[m] for m in avail}

sns.boxplot(
    data=plot_df, x="Method", y="total_tokens",
    order=order, palette=pal, width=0.5,
    flierprops=dict(marker="o", markersize=4, alpha=0.5),
    ax=ax,
)
sns.stripplot(
    data=plot_df, x="Method", y="total_tokens",
    order=order, palette=pal, size=5, alpha=0.6, jitter=True, ax=ax,
)

ax.set_title("Per-Task Token Distribution", fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("Total Tokens")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.tick_params(axis="x", labelrotation=10)

plt.tight_layout()
plt.savefig(RESULTS / "plot_token_distribution.png", bbox_inches="tight")
plt.show()

/var/folders/c4/98dp1lqj4kq9n2s3k05mvq7m0000gn/T/ipykernel_80898/2257098803.py:8: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
/var/folders/c4/98dp1lqj4kq9n2s3k05mvq7m0000gn/T/ipykernel_80898/2257098803.py:14: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.stripplot(
/var/folders/c4/98dp1lqj4kq9n2s3k05mvq7m0000gn/T/ipykernel_80898/2257098803.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Token Breakdown: Input vs Output

Input tokens dominate cost for methods that re-feed the entire program in repair prompts (self-debugging). Output tokens are similar across methods.

In [9]:
fig, ax = plt.subplots(figsize=(9, 4))

x = np.arange(len(avail))
w = 0.35

avg_in  = [df[df["method"]==m]["total_input_tokens"].mean()  for m in avail]
avg_out = [df[df["method"]==m]["total_output_tokens"].mean() for m in avail]

b1 = ax.bar(x - w/2, avg_in,  w, label="Input tokens",  color="#5b9bd5", edgecolor="white")
b2 = ax.bar(x + w/2, avg_out, w, label="Output tokens", color="#ed7d31", edgecolor="white")

ax.set_xticks(x)
ax.set_xticklabels([METHOD_LABELS[m] for m in avail], rotation=10)
ax.set_ylabel("Avg Tokens")
ax.set_title("Avg Input vs Output Tokens per Task", fontweight="bold")
ax.legend()

for bar in list(b1) + list(b2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 10, f"{h:,.0f}",
            ha="center", va="bottom", fontsize=8.5)

plt.tight_layout()
plt.savefig(RESULTS / "plot_input_output_tokens.png", bbox_inches="tight")
plt.show()

/var/folders/c4/98dp1lqj4kq9n2s3k05mvq7m0000gn/T/ipykernel_80898/4144478804.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Token Overhead vs Direct Generation

How much more (or less) does each method cost relative to the `direct_gen` baseline?

In [10]:
direct_avg = df[df["method"]=="direct_gen"]["total_tokens"].mean() if "direct_gen" in avail else None

if direct_avg:
    fig, ax = plt.subplots(figsize=(8, 3.5))

    non_direct = [m for m in avail if m != "direct_gen"]
    overheads  = [
        (df[df["method"]==m]["total_tokens"].mean() / direct_avg - 1) * 100
        for m in non_direct
    ]
    bar_colors = ["#c00000" if o > 0 else "#375623" for o in overheads]

    bars = ax.barh([METHOD_LABELS[m] for m in non_direct], overheads,
                   color=bar_colors, edgecolor="white", height=0.5)
    ax.axvline(0, color="grey", linewidth=1)
    ax.set_xlabel("Token overhead vs Direct Gen (%)")
    ax.set_title("Token Overhead vs Direct Generation", fontweight="bold")
    ax.xaxis.set_major_formatter(mticker.PercentFormatter())

    for bar, val in zip(bars, overheads):
        xpos = val + (1.5 if val >= 0 else -1.5)
        ax.text(xpos, bar.get_y() + bar.get_height()/2,
                f"{val:+.1f}%", va="center", ha="left" if val >= 0 else "right",
                fontsize=10, fontweight="bold")

    plt.tight_layout()
    plt.savefig(RESULTS / "plot_overhead.png", bbox_inches="tight")
    plt.show()
else:
    print("direct_gen results not found — skipping overhead chart")

/var/folders/c4/98dp1lqj4kq9n2s3k05mvq7m0000gn/T/ipykernel_80898/2999689653.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Token Cost by Call Type

Breaks down where tokens are spent inside each method (generation, planning, explanation, repair, etc.).

In [11]:
if not df_events.empty:
    df_events["total_tokens"] = df_events["input_tokens"] + df_events["output_tokens"]

    # Per-method average token cost by call_type
    ct_avg = (
        df_events.groupby(["method", "call_type"])["total_tokens"]
        .sum()
        .reset_index()
        .pivot(index="method", columns="call_type", values="total_tokens")
        .fillna(0)
    )
    # Normalize per task
    task_counts = df.groupby("method")["task_id"].count()
    ct_avg = ct_avg.div(task_counts, axis=0).reindex([m for m in avail if m in ct_avg.index])
    ct_avg.index = [METHOD_LABELS[m] for m in ct_avg.index]

    # Drop near-zero columns
    ct_avg = ct_avg.loc[:, ct_avg.max() > 10]

    cmap = plt.get_cmap("tab10")
    call_colors = {c: cmap(i) for i, c in enumerate(ct_avg.columns)}

    ax = ct_avg.plot(
        kind="bar", stacked=True, figsize=(10, 4.5),
        color=[call_colors[c] for c in ct_avg.columns],
        edgecolor="white", linewidth=0.5,
    )
    ax.set_title("Avg Token Cost per Task by Call Type", fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel("Avg Tokens")
    ax.tick_params(axis="x", labelrotation=10)
    ax.legend(title="Call type", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=9)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

    plt.tight_layout()
    plt.savefig(RESULTS / "plot_call_type_tokens.png", bbox_inches="tight")
    plt.show()
else:
    print("No token_events.jsonl found — skipping call-type breakdown")

/var/folders/c4/98dp1lqj4kq9n2s3k05mvq7m0000gn/T/ipykernel_80898/2185248895.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Retry Behaviour

How many retries does each method use on average, and does retrying help?

In [12]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# ── (a) Retry count distribution ────────────────────────────────────
ax = axes[0]
for m in avail:
    sub = df[df["method"]==m]["num_retries"]
    counts = sub.value_counts().sort_index()
    ax.bar(counts.index + [avail.index(m)*0.15], counts.values,
           width=0.13, label=METHOD_LABELS[m], color=PALETTE[m], edgecolor="white")
ax.set_title("Retry Count Distribution", fontweight="bold")
ax.set_xlabel("Number of retries")
ax.set_ylabel("Tasks")
ax.set_xticks(sorted(df["num_retries"].unique()))
ax.legend(fontsize=9)

# ── (b) Tokens vs retries scatter ───────────────────────────────────
ax = axes[1]
for m in avail:
    sub = df[df["method"]==m]
    ax.scatter(sub["num_retries"], sub["total_tokens"],
               label=METHOD_LABELS[m], color=PALETTE[m], alpha=0.7, s=50)
ax.set_title("Tokens vs Retries", fontweight="bold")
ax.set_xlabel("Retries")
ax.set_ylabel("Total tokens")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(RESULTS / "plot_retries.png", bbox_inches="tight")
plt.show()

/var/folders/c4/98dp1lqj4kq9n2s3k05mvq7m0000gn/T/ipykernel_80898/3211872791.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. StateGen: Memory Reuse

Tracks pattern reuse across tasks — a key efficiency advantage of StateGen that compounds over more tasks.

In [13]:
if "stategen" in avail and not df_events.empty:
    sg_events = df_events[df_events["method"]=="stategen"].copy()
    sg_events = sg_events.sort_values("timestamp").reset_index(drop=True)

    # Count memory hits from attempt_details
    sg_df = df[df["method"]=="stategen"].copy()
    memory_hits = []
    for _, row in sg_df.iterrows():
        hits = sum(
            1 for d in (row.get("attempt_details") or [])
            if d.get("from_memory", False)
        )
        memory_hits.append({"task_id": row["task_id"], "memory_hits": hits})
    mem_df = pd.DataFrame(memory_hits)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # (a) Memory hits per task
    ax = axes[0]
    ax.bar(range(len(mem_df)), mem_df["memory_hits"], color=PALETTE["stategen"], edgecolor="white")
    ax.set_title("Memory Hits per Task (StateGen)", fontweight="bold")
    ax.set_xlabel("Task index")
    ax.set_ylabel("State patterns reused")
    ax.set_xticks(range(len(mem_df)))
    ax.set_xticklabels([t.split("/")[-1] for t in mem_df["task_id"]], rotation=45, fontsize=8)

    # (b) Cumulative memory patterns stored
    ax = axes[1]
    # Approximate: each stategen task writes ~2 patterns (1 per state)
    memory_path = ROOT / "memory" / "patterns.json"
    if memory_path.exists():
        with open(memory_path) as f:
            patterns = json.load(f)
        pat_df = pd.DataFrame(patterns)
        ax.bar(range(len(pat_df)), pat_df["success_count"],
               color=PALETTE["stategen"], edgecolor="white")
        ax.set_title("Pattern Reuse Counts (Memory Store)", fontweight="bold")
        ax.set_xlabel("Pattern index")
        ax.set_ylabel("Times reused")
        total_reuse = pat_df["success_count"].sum() - len(pat_df)
        ax.set_title(f"Memory Patterns: {len(pat_df)} stored, {total_reuse} reuses",
                     fontweight="bold")
    else:
        ax.text(0.5, 0.5, "memory/patterns.json not found",
                ha="center", va="center", transform=ax.transAxes)

    plt.tight_layout()
    plt.savefig(RESULTS / "plot_memory_reuse.png", bbox_inches="tight")
    plt.show()
else:
    print("No stategen results or token_events — skipping memory plot")

/var/folders/c4/98dp1lqj4kq9n2s3k05mvq7m0000gn/T/ipykernel_80898/2822752020.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Efficiency Summary Table

In [14]:
direct_avg = df[df["method"]=="direct_gen"]["total_tokens"].mean() if "direct_gen" in avail else None

rows = []
for m in avail:
    sub = df[df["method"]==m]
    avg_tok = sub["total_tokens"].mean()
    overhead = (avg_tok / direct_avg - 1) * 100 if direct_avg else float("nan")
    
    passed = sub[sub["passed"]]
    failed = sub[~sub["passed"]]
    wasted = failed["total_tokens"].sum() / sub["total_tokens"].sum() if sub["total_tokens"].sum() > 0 else 1.0

    rows.append({
        "Method":         METHOD_LABELS[m],
        "Tasks":          len(sub),
        "Pass@1":         f"{sub['passed'].mean():.1%}",
        "Avg Tokens":     f"{avg_tok:,.0f}",
        "Tok/Correct":    f"{passed['total_tokens'].mean():,.0f}" if len(passed) else "—",
        "Wasted %":       f"{wasted:.1%}",
        "Overhead vs DG": f"{overhead:+.1f}%" if not np.isnan(overhead) and m != 'direct_gen' else "baseline",
        "Avg Retries":    f"{sub['num_retries'].mean():.2f}",
        "Avg Time (s)":   f"{sub['wall_time'].mean():.1f}s",
    })

tbl = pd.DataFrame(rows).set_index("Method")

# Pretty display
display(tbl.style
    .set_caption("Experiment Summary")
    .set_table_styles([{
        "selector": "caption",
        "props": "font-size: 1.2em; font-weight: bold; text-align: left; padding-bottom: 8px;"
    }])
    .set_properties(**{"text-align": "right"})
    .set_properties(subset=["Method"] if "Method" in tbl.columns else [],
                    **{"text-align": "left"})
)

,Tasks,Pass@1,Avg Tokens,Tok/Correct,Wasted %,Overhead vs DG,Avg Retries,Avg Time (s)
Method,,,,,,,,
Direct Gen,145,23.4%,604,593,76.9%,baseline,0.00,19.0s
Self-Planning,145,29.0%,"3,178","1,674",84.7%,+426.5%,3.02,86.8s
Self-Debugging,145,30.3%,"6,560","1,687",92.2%,+986.9%,1.48,140.0s
StateGen (ours),145,29.7%,"3,093","2,445",76.6%,+412.5%,1.80,74.4s


## 9. Per-Task Breakdown

Heatmap of token usage per task across methods — reveals which tasks are consistently expensive.

In [15]:
# Pivot to task × method
pivot = df.pivot_table(index="task_id", columns="method", values="total_tokens", aggfunc="mean")
pivot = pivot[[m for m in avail if m in pivot.columns]]  # order columns
pivot.columns = [METHOD_LABELS[m] for m in pivot.columns]
pivot.index = [t.split("/")[-1] for t in pivot.index]   # short task ids

fig_h = max(4, len(pivot) * 0.38)
fig, ax = plt.subplots(figsize=(max(7, len(avail) * 2.2), fig_h))
sns.heatmap(
    pivot, annot=True, fmt=".0f", cmap="YlOrRd",
    linewidths=0.4, linecolor="white",
    cbar_kws={"label": "Total Tokens"},
    ax=ax,
)
ax.set_title("Token Usage per Task × Method", fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("Task")
ax.tick_params(axis="x", labelrotation=15)
ax.tick_params(axis="y", labelrotation=0)

plt.tight_layout()
plt.savefig(RESULTS / "plot_task_heatmap.png", bbox_inches="tight")
plt.show()

/var/folders/c4/98dp1lqj4kq9n2s3k05mvq7m0000gn/T/ipykernel_80898/3468234357.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. Wall Time

Latency matters for interactive use. Methods with more LLM calls have higher wall times.

In [16]:
fig, ax = plt.subplots(figsize=(9, 4))

plot_df = df[df["method"].isin(avail)].copy()
plot_df["Method"] = plot_df["method"].map(METHOD_LABELS)
order = [METHOD_LABELS[m] for m in avail]
pal   = {METHOD_LABELS[m]: PALETTE[m] for m in avail}

sns.boxplot(
    data=plot_df, x="Method", y="wall_time",
    order=order, palette=pal, width=0.5, ax=ax,
)

ax.set_title("Wall Time per Task (seconds)", fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("Wall time (s)")
ax.tick_params(axis="x", labelrotation=10)

plt.tight_layout()
plt.savefig(RESULTS / "plot_wall_time.png", bbox_inches="tight")
plt.show()

/var/folders/c4/98dp1lqj4kq9n2s3k05mvq7m0000gn/T/ipykernel_80898/2965546778.py:8: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(
/var/folders/c4/98dp1lqj4kq9n2s3k05mvq7m0000gn/T/ipykernel_80898/2965546778.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Notes

- **Pass@1** is the fraction of tasks solved. Without `--remote_eval`, this uses `quick_check` (syntax + import check only) as a proxy — not the full BigCodeBench unittest suite.
- To get official pass@1, run the sanitize + Docker evaluate pipeline, then re-run this notebook.
- The memory reuse plots will show richer data as more tasks accumulate in `memory/patterns.json`.